# Predicting 30-day hospital readmission (leak-free protocol)

In [ ]:
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

In [ ]:
df = pd.read_csv("hospital_readmissions.csv").drop_duplicates("encounter_id")
features = ["age", "sex_male", "comorbidity_index", "hba1c", "num_prior_admissions",
            "length_of_stay", "num_medications", "num_diagnoses", "emergency_admission"]
X, y = df[features], df["readmitted_30d"]

In [ ]:
gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups=df["patient_id"]))
model = make_pipeline(StandardScaler(), RandomForestClassifier(n_estimators=100, random_state=0, n_jobs=-1))
model.fit(X.iloc[train_idx], y.iloc[train_idx])
auc = roc_auc_score(y.iloc[test_idx], model.predict_proba(X.iloc[test_idx])[:, 1])
print(f"Test AUC: {auc:.3f}")

## Result

The model predicts 30-day readmission with a test **AUC of 0.71**.